In [10]:
import re
import os
import json
import time
import ast
import configparser
from pathlib import Path

import pandas as pd
from openai import OpenAI

In [11]:
# ----------------------------
# File paths
# ----------------------------
CONFIG_PATH = Path(r"C:\Users\lmaefos\Code Stuffs\CDE_detective\CDE_ID_detective_revamp\config_prestep.ini")
ENV_PATH = Path(r"C:\Users\lmaefos\Code Stuffs\CDE_detective\CDE_ID_detective_revamp\.env")
CRF_JSON_PATH = Path(r"C:\Users\lmaefos\Code Stuffs\CDE_detective\CDE_ID_detective_revamp\KnowledgeBase\CRF_descriptions.json")

# ----------------------------
# Helpers
# ----------------------------
def load_env_file(env_path: Path) -> dict:
    env_vars = {}
    with open(env_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#") or "=" not in line:
                continue
            key, value = line.split("=", 1)
            env_vars[key.strip()] = value.strip().strip('"').strip("'")
    return env_vars


def build_crf_reference_text(crf_json: dict) -> str:
    """
    Turn CRF_descriptions.json into a compact prompt-ready reference block.
    """
    lines = []
    for crf in crf_json.get("CRFs", []):
        name = crf.get("name", "").strip()
        abbreviations = ", ".join(crf.get("abbreviations", []))
        description = crf.get("description", "").strip()

        lines.append(f"Official CRF Name: {name}")
        if abbreviations:
            lines.append(f"Abbreviations: {abbreviations}")
        if description:
            lines.append(f"Description: {description}")
        lines.append("")

    return "\n".join(lines).strip()

# ----------------------------
# Load config
# ----------------------------
config = configparser.ConfigParser()
config.read(CONFIG_PATH, encoding="utf-8")

input_file = config["Files"]["input_file"]
input_worksheet = config["Files"]["input_worksheet"]
output_file = config["Files"]["output_file"]

crf_column = config["Columns"]["crf_column"]
variable_column = config["Columns"]["variable_column"]
description_column = config["Columns"]["description_column"]

crf_id_prestep_instruction = config["Instructions"]["crf_id_prestep"]
form_harmonizer_instruction = config["Instructions"]["form_harmonizer"]
matching_instruction = config["Instructions"]["matching_instruction"]

# Optional model overrides from config
PRESTEP_MODEL = config.get("Models", "prestep_model", fallback="gpt-4.1-mini")
HARMONIZER_MODEL = config.get("Models", "harmonizer_model", fallback="gpt-4.1-mini")
MATCHING_MODEL = config.get("Models", "matching_model", fallback="gpt-4.1-mini")

# ----------------------------
# Load .env + client
# ----------------------------
env_vars = load_env_file(ENV_PATH)
api_key = env_vars.get("OPENAI_API_KEY") or env_vars.get("api_key")

if not api_key:
    raise ValueError("No OpenAI API key found in .env. Expected OPENAI_API_KEY or api_key.")

client = OpenAI(api_key=api_key)

# ----------------------------
# Load local CRF KB
# ----------------------------
with open(CRF_JSON_PATH, "r", encoding="utf-8") as f:
    crf_kb = json.load(f)

crf_reference_text = build_crf_reference_text(crf_kb)

print("✅ Setup loaded successfully.")
print(f"Config path: {CONFIG_PATH}")
print(f".env path: {ENV_PATH}")
print(f"CRF JSON path: {CRF_JSON_PATH}")
print(f"Input file: {input_file}")
print(f"Input worksheet: {input_worksheet}")
print(f"Output file: {output_file}")
print(f"API key loaded: {'Yes' if api_key else 'No'}")
print(f"OpenAI client created: {'Yes' if client else 'No'}")
print(f"Loaded {len(crf_kb.get('CRFs', []))} CRF definitions.")
print(f"Prestep model: {PRESTEP_MODEL}")
print(f"Harmonizer model: {HARMONIZER_MODEL}")
print(f"Matching model: {MATCHING_MODEL}")

print("\n--- CRF reference preview ---")
print(crf_reference_text[:1200] + ("..." if len(crf_reference_text) > 1200 else ""))

✅ Setup loaded successfully.
Config path: C:\Users\lmaefos\Code Stuffs\CDE_detective\CDE_ID_detective_revamp\config_prestep.ini
.env path: C:\Users\lmaefos\Code Stuffs\CDE_detective\CDE_ID_detective_revamp\.env
CRF JSON path: C:\Users\lmaefos\Code Stuffs\CDE_detective\CDE_ID_detective_revamp\KnowledgeBase\CRF_descriptions.json
Input file: in\Testfile.xlsx
Input worksheet: Sheet1
Output file: out\Testfile_2026-04-29.xlsx
API key loaded: Yes
OpenAI client created: Yes
Loaded 24 CRF definitions.
Prestep model: gpt-4.1-mini
Harmonizer model: gpt-4.1-mini
Matching model: gpt-4.1-mini

--- CRF reference preview ---
Official CRF Name: Brief Pain Inventory (BPI)
Abbreviations: BPI, Brief Pain Inventory, B.P.I.
Description: A self-report questionnaire measuring pain severity and interference with daily activities.

Official CRF Name: BPI Pain Interference
Abbreviations: BPI Interference, Pain Interference, BPI-PI
Description: Assesses how pain impacts various aspects of daily life, including mo

In [12]:
# ----------------------------
# Responses API prestep call functions
# ----------------------------
def build_prestep_input(crf_name, variable_name, description_text, acronym_hint=""):
    acronym_block = ""
    if acronym_hint:
        acronym_block = f"\nDeterministic acronym hint from variable name: {acronym_hint}\n"

    return f"""
Study row context:
- Original CRF/Form Name: {crf_name}
- Variable Name: {variable_name}
- Description / Field Label: {description_text}
{acronym_block}
Official HEAL Core CRF reference:
{crf_reference_text}

Task:
Use the study row context, deterministic acronym hint when present, and the official HEAL Core CRF reference to identify
the most likely HEAL Core CRF or state that no confident HEAL Core CRF match exists.

Return JSON with exactly these keys:
- CRF
- Rationale
""".strip()


def call_prestep_responses(crf_name, variable_name, description_text):
    """
    Single Responses API call for prestep CRF identification.
    Returns plain text output.
    """
    user_input = build_prestep_input(crf_name, variable_name, description_text)

    response = client.responses.create(
        model=PRESTEP_MODEL,
        instructions=crf_id_prestep_instruction,
        input=user_input
    )

    return response.output_text.strip()


def call_prestep_responses_with_retry(crf_name, variable_name, description_text, max_retries=3, base_sleep_seconds=2):
    """
    Safe wrapper around the Responses API prestep call.
    Returns a dictionary so failures do not crash the whole run.
    """
    last_error = ""

    for attempt in range(1, max_retries + 1):
        try:
            output_text = call_prestep_responses(crf_name, variable_name, description_text)

            return {
                "Full Response": output_text,
                "Prestep Run Status": "Reviewed",
                "Prestep Attempts": attempt,
                "Prestep Error": ""
            }

        except Exception as e:
            last_error = str(e)
            print(f"[warning] prestep failed on attempt {attempt}/{max_retries}: {last_error}")

            if attempt < max_retries:
                sleep_time = base_sleep_seconds * attempt
                print(f"Retrying in {sleep_time} seconds...")
                time.sleep(sleep_time)

    return {
        "Full Response": "",
        "Prestep Run Status": "ERROR",
        "Prestep Attempts": max_retries,
        "Prestep Error": last_error
    }


print("✅ Responses API prestep call functions loaded.")

✅ Responses API prestep call functions loaded.


In [13]:
# ----------------------------
# Parse / normalize Full Response
# ----------------------------
def parse_full_response_cell(value):
    """
    Robust parser for Full Response values that may look like:
    1. Proper JSON object string
    2. Quoted JSON string with escaped quotes
    3. Multi-line JSON-like string
    """
    if pd.isna(value):
        return {
            "Refined CRF Name": "",
            "Rationale": "",
            "Parsed Full Response": "",
            "Parse Status": "EMPTY",
            "Parse Error": ""
        }

    raw = str(value).strip()

    if not raw:
        return {
            "Refined CRF Name": "",
            "Rationale": "",
            "Parsed Full Response": "",
            "Parse Status": "EMPTY",
            "Parse Error": ""
        }

    errors = []

    # Attempt 1: direct JSON parse
    try:
        parsed = json.loads(raw)

        if isinstance(parsed, dict):
            return {
                "Refined CRF Name": str(parsed.get("CRF", "")).strip(),
                "Rationale": str(parsed.get("Rationale", "")).strip(),
                "Parsed Full Response": json.dumps(parsed, ensure_ascii=False),
                "Parse Status": "PARSED_JSON",
                "Parse Error": ""
            }

        # If first parse returns a string, try parsing that as JSON again
        if isinstance(parsed, str):
            try:
                parsed2 = json.loads(parsed)
                if isinstance(parsed2, dict):
                    return {
                        "Refined CRF Name": str(parsed2.get("CRF", "")).strip(),
                        "Rationale": str(parsed2.get("Rationale", "")).strip(),
                        "Parsed Full Response": json.dumps(parsed2, ensure_ascii=False),
                        "Parse Status": "PARSED_DOUBLE_JSON",
                        "Parse Error": ""
                    }
            except Exception as e2:
                errors.append(f"double_json: {e2}")

    except Exception as e1:
        errors.append(f"json: {e1}")

    # Attempt 2: Python literal parse
    try:
        literal = ast.literal_eval(raw)

        if isinstance(literal, dict):
            return {
                "Refined CRF Name": str(literal.get("CRF", "")).strip(),
                "Rationale": str(literal.get("Rationale", "")).strip(),
                "Parsed Full Response": json.dumps(literal, ensure_ascii=False),
                "Parse Status": "PARSED_LITERAL_DICT",
                "Parse Error": ""
            }

        if isinstance(literal, str):
            try:
                parsed3 = json.loads(literal)
                if isinstance(parsed3, dict):
                    return {
                        "Refined CRF Name": str(parsed3.get("CRF", "")).strip(),
                        "Rationale": str(parsed3.get("Rationale", "")).strip(),
                        "Parsed Full Response": json.dumps(parsed3, ensure_ascii=False),
                        "Parse Status": "PARSED_LITERAL_TO_JSON",
                        "Parse Error": ""
                    }
            except Exception as e3:
                errors.append(f"literal_to_json: {e3}")

    except Exception as e4:
        errors.append(f"literal: {e4}")

    # Fallback
    return {
        "Refined CRF Name": "",
        "Rationale": "",
        "Parsed Full Response": raw,
        "Parse Status": "PARSE_FAILED",
        "Parse Error": " | ".join(errors)
    }


print("✅ Full Response parser loaded.")

✅ Full Response parser loaded.


In [14]:
# ----------------------------
# Full dataframe prestep runner using Responses API + parsing
# ----------------------------
def run_prestep_responses(df, chunk_size=20, checkpoint_every=20, checkpoint_path=None):
    """
    Run prestep generation across the dataframe using the Responses API.

    Writes:
    - Full Response
    - Prestep Run Status
    - Prestep Attempts
    - Prestep Error
    - Refined CRF Name
    - Rationale
    - Parsed Full Response
    - Parse Status
    - Parse Error
    """
    working_df = df.copy()

    # Initialize output columns if missing
    output_cols = [
        "Full Response",
        "Prestep Run Status",
        "Prestep Attempts",
        "Prestep Error",
        "Refined CRF Name",
        "Rationale",
        "Parsed Full Response",
        "Parse Status",
        "Parse Error"
    ]

    for col in output_cols:
        if col not in working_df.columns:
            working_df[col] = ""

    processed_count = 0

    for start in range(0, len(working_df), chunk_size):
        chunk = working_df.iloc[start:start + chunk_size]

        print(f"\nProcessing rows {start} to {start + len(chunk) - 1}...")

        for idx, row in chunk.iterrows():
            crf_value = str(row[crf_column]) if pd.notna(row[crf_column]) else ""
            var_value = str(row[variable_column]) if pd.notna(row[variable_column]) else ""
            desc_value = str(row[description_column]) if pd.notna(row[description_column]) else ""

            result = call_prestep_responses_with_retry(
                crf_name=crf_value,
                variable_name=var_value,
                description_text=desc_value,
                max_retries=3,
                base_sleep_seconds=2
            )

            # Save raw prestep call outputs
            working_df.at[idx, "Full Response"] = result["Full Response"]
            working_df.at[idx, "Prestep Run Status"] = result["Prestep Run Status"]
            working_df.at[idx, "Prestep Attempts"] = result["Prestep Attempts"]
            working_df.at[idx, "Prestep Error"] = result["Prestep Error"]

            # Parse immediately so downstream steps get clean columns
            parsed = parse_full_response_cell(result["Full Response"])

            working_df.at[idx, "Refined CRF Name"] = parsed["Refined CRF Name"]
            working_df.at[idx, "Rationale"] = parsed["Rationale"]
            working_df.at[idx, "Parsed Full Response"] = parsed["Parsed Full Response"]
            working_df.at[idx, "Parse Status"] = parsed["Parse Status"]
            working_df.at[idx, "Parse Error"] = parsed["Parse Error"]

            processed_count += 1

            if checkpoint_path and processed_count % checkpoint_every == 0:
                working_df.to_excel(checkpoint_path, index=False)
                print(f"Checkpoint saved after {processed_count} rows to: {checkpoint_path}")

    return working_df


print("✅ run_prestep_responses() loaded.")

✅ run_prestep_responses() loaded.


In [15]:
# ----------------------------
# Acronym finder + shared JSON parsing helpers
# ----------------------------
def load_acronym_map_from_config(config, section="Acronyms"):
    """
    Load acronym mappings from config_prestep.ini.

    Example config section:
    [Acronyms]
    gad7 = GAD-7
    bpi = BPI
    promis = PROMIS
    """
    if not config.has_section(section):
        print(f"⚠️ No [{section}] section found in config. Acronym finder will return blanks.")
        return {}

    acronym_map = {
        key.strip().lower(): value.strip()
        for key, value in config.items(section)
        if key.strip() and value.strip()
    }

    print(f"✅ Loaded {len(acronym_map)} acronym mappings from [{section}].")
    return acronym_map


ACRONYM_MAP = load_acronym_map_from_config(config)


def detect_heal_cde_acronym(variable_name, acronym_map=ACRONYM_MAP):
    """
    Detect likely HEAL CDE instrument acronym from a variable name.

    Uses acronym mappings loaded from config_prestep.ini.
    Returns the configured acronym label, such as 'BPI' or 'GAD-2'.
    """
    if pd.isna(variable_name):
        return ""

    if not acronym_map:
        return ""

    v = str(variable_name).strip().lower()

    # Compact version helps catch CDE-style variable names:
    # Example: GAD2FeelNervScale -> gad2feelnervscale
    compact = re.sub(r"[^a-z0-9]", "", v)

    # Sort longest first so gad7/gad2/phq9 match before gad/phq
    for key in sorted(acronym_map.keys(), key=len, reverse=True):
        key_compact = re.sub(r"[^a-z0-9]", "", key.lower())

        if compact.startswith(key_compact) or key_compact in compact:
            return acronym_map[key]

    return ""


print("✅ Config-driven acronym finder loaded.")


def parse_json_object_from_text(text):
    """
    Generic helper for extracting a JSON object from model text output.
    Handles:
    - normal JSON objects
    - quoted JSON strings
    - escaped strings
    """
    if text is None:
        return None

    raw = str(text).strip()
    if not raw:
        return None

    # Attempt 1: direct JSON
    try:
        parsed = json.loads(raw)
        if isinstance(parsed, dict):
            return parsed
        if isinstance(parsed, str):
            parsed2 = json.loads(parsed)
            if isinstance(parsed2, dict):
                return parsed2
    except Exception:
        pass

    # Attempt 2: Python literal -> maybe dict or JSON string
    try:
        literal = ast.literal_eval(raw)
        if isinstance(literal, dict):
            return literal
        if isinstance(literal, str):
            parsed3 = json.loads(literal)
            if isinstance(parsed3, dict):
                return parsed3
    except Exception:
        pass

    return None


print("✅ Acronym finder + shared JSON helpers loaded.")

✅ Loaded 19 acronym mappings from [Acronyms].
✅ Config-driven acronym finder loaded.
✅ Acronym finder + shared JSON helpers loaded.


In [16]:
# ----------------------------
# Responses API form harmonizer
# ----------------------------
def batcher(seq, size=20):
    """Yield successive size-sized chunks from seq."""
    for pos in range(0, len(seq), size):
        yield seq[pos:pos + size]


def call_form_harmonizer_responses(batch):
    """
    Call the Responses API to harmonize a batch of refined CRF names.
    Expects a JSON object with a top-level 'mapping' key.
    """
    user_input = json.dumps(batch, ensure_ascii=False, indent=2)

    response = client.responses.create(
        model=HARMONIZER_MODEL,
        instructions=form_harmonizer_instruction,
        input=user_input
    )

    raw_text = response.output_text.strip()
    parsed = parse_json_object_from_text(raw_text)

    if not parsed:
        raise ValueError(f"Could not parse harmonizer output as JSON.\nRaw output:\n{raw_text}")

    if "mapping" in parsed and isinstance(parsed["mapping"], dict):
        mapping = parsed["mapping"]
    elif isinstance(parsed, dict):
        mapping = parsed
    else:
        raise ValueError(f"Harmonizer output JSON did not contain a usable mapping.\nParsed output:\n{parsed}")

    return mapping, raw_text


def call_form_harmonizer_with_retry(batch, max_retries=3, base_sleep_seconds=2):
    """
    Safe wrapper around the Responses API harmonizer call.
    """
    last_error = ""

    for attempt in range(1, max_retries + 1):
        try:
            mapping, raw_text = call_form_harmonizer_responses(batch)
            return {
                "mapping": mapping,
                "raw_harmonizer_response": raw_text,
                "harmonizer_status": "Reviewed",
                "harmonizer_attempts": attempt,
                "harmonizer_error": ""
            }

        except Exception as e:
            last_error = str(e)
            print(f"[warning] harmonizer failed on attempt {attempt}/{max_retries}: {last_error}")

            if attempt < max_retries:
                sleep_time = base_sleep_seconds * attempt
                print(f"Retrying in {sleep_time} seconds...")
                time.sleep(sleep_time)

    return {
        "mapping": {},
        "raw_harmonizer_response": "",
        "harmonizer_status": "ERROR",
        "harmonizer_attempts": max_retries,
        "harmonizer_error": last_error
    }


def run_form_harmonizer(refined_df, batch_size=20):
    """
    Harmonize refined CRF names into canonical CRF names using batched Responses API calls.
    """
    working_df = refined_df.copy()

    # Build and dedupe the payload
    seen = set()
    unique_entries = []

    for orig, rat in zip(working_df["Refined CRF Name"], working_df["Rationale"]):
        key = (_norm_value(orig), _norm_value(rat))
        if key not in seen:
            seen.add(key)
            unique_entries.append({
                "original": _norm_value(orig),
                "rationale": _norm_value(rat)
            })

    if not unique_entries:
        working_df["Canonical CRF Name"] = working_df["Refined CRF Name"]
        print("[Harmonizer] No entries to harmonize, using identity mapping.")
        return working_df

    combined_mapping = {}

    for batch_num, batch in enumerate(batcher(unique_entries, size=batch_size), start=1):
        print(f"\n[Harmonizer] Sending batch {batch_num} of {len(batch)}:")
        for entry in batch:
            print("   ", entry)

        result = call_form_harmonizer_with_retry(batch, max_retries=3, base_sleep_seconds=2)
        mapping = result["mapping"]

        if not mapping:
            print("[Harmonizer] Empty mapping returned; defaulting this batch to identity mapping.")
            mapping = {entry["original"]: entry["original"] for entry in batch}

        print("\n[Harmonizer] Parsed mapping (original → harmonized):")
        for orig, canon in mapping.items():
            print(f"   '{orig}' -> '{canon}'")

        combined_mapping.update(mapping)

    working_df["Canonical CRF Name"] = working_df["Refined CRF Name"].map(
        lambda x: combined_mapping.get(_norm_value(x), _norm_value(x))
    )

    print("\n[Harmonizer] Final Canonical CRF Name results:")
    print(
        working_df[["Refined CRF Name", "Canonical CRF Name"]]
        .drop_duplicates()
        .reset_index(drop=True)
    )

    return working_df


def _norm_value(value):
    if pd.isna(value):
        return ""
    return str(value).strip()


print("✅ Responses API form harmonizer loaded.")

✅ Responses API form harmonizer loaded.


In [17]:
# ----------------------------
# Responses API HEAL Core CRF matcher
# ----------------------------
def build_heal_match_input(full_prestep_response, acronym_hint=None):
    """
    Build the user input for the HEAL Core CRF matching step.
    """
    acronym_block = ""
    if acronym_hint:
        acronym_block = f"\nAcronym hint detected from variable name: {acronym_hint}\n"

    return (
        f"Prestep output:\n{full_prestep_response}\n"
        f"{acronym_block}\n"
        "Please respond in strict JSON with keys "
        "\"heal_core_crf\", \"confidence\", and \"rationale\". "
        "Do not wrap in markdown or add any extra fields."
    ).strip()


def call_heal_match_responses(full_prestep_response, acronym_hint=None):
    """
    Single Responses API call for HEAL Core CRF matching.
    Returns a parsed dictionary.
    """
    user_input = build_heal_match_input(full_prestep_response, acronym_hint)

    response = client.responses.create(
        model=MATCHING_MODEL,
        instructions=matching_instruction,
        input=user_input
    )

    raw_text = response.output_text.strip()
    parsed = parse_json_object_from_text(raw_text)

    if not parsed:
        raise ValueError(f"Could not parse HEAL match output as JSON.\nRaw output:\n{raw_text}")

    return parsed, raw_text


def call_heal_match_with_retry(full_prestep_response, acronym_hint=None, max_retries=3, base_sleep_seconds=2):
    """
    Safe wrapper around the Responses API HEAL Core CRF match call.
    """
    last_error = ""

    for attempt in range(1, max_retries + 1):
        try:
            parsed, raw_text = call_heal_match_responses(full_prestep_response, acronym_hint)

            match = str(parsed.get("heal_core_crf", "No CRF match")).strip()
            conf = str(parsed.get("confidence", "")).strip()
            rationale = str(parsed.get("rationale", "")).strip()

            return {
                "HEAL Core CRF Match": match,
                "Prestep CRF Confidence": conf,
                "Match Rationale": rationale,
                "Raw HEAL Match Response": raw_text,
                "HEAL Match Status": "Reviewed",
                "HEAL Match Attempts": attempt,
                "HEAL Match Error": ""
            }

        except Exception as e:
            last_error = str(e)
            print(f"[warning] HEAL match failed on attempt {attempt}/{max_retries}: {last_error}")

            if attempt < max_retries:
                sleep_time = base_sleep_seconds * attempt
                print(f"Retrying in {sleep_time} seconds...")
                time.sleep(sleep_time)

    return {
        "HEAL Core CRF Match": "No CRF match",
        "Prestep CRF Confidence": "",
        "Match Rationale": "",
        "Raw HEAL Match Response": "",
        "HEAL Match Status": "ERROR",
        "HEAL Match Attempts": max_retries,
        "HEAL Match Error": last_error
    }


def run_heal_match(df, chunk_size=20):
    """
    Run HEAL Core CRF matching over the dataframe.

    Requires:
    - Full Response
    - CDE Acronym Finder

    Optional:
    - Parse Status
      If present, rows with parsing failures will be skipped before calling the LLM.
    """
    working_df = df.copy()

    output_cols = [
        "HEAL Core CRF Match",
        "Prestep CRF Confidence",
        "Match Rationale",
        "Raw HEAL Match Response",
        "HEAL Match Status",
        "HEAL Match Attempts",
        "HEAL Match Error"
    ]

    for col in output_cols:
        if col not in working_df.columns:
            working_df[col] = ""

    for start in range(0, len(working_df), chunk_size):
        chunk = working_df.iloc[start:start + chunk_size]
        print(f"\n[HEAL Match] Processing rows {start} to {start + len(chunk) - 1}...")

        for idx, row in chunk.iterrows():
            full_response = str(row["Full Response"]) if pd.notna(row["Full Response"]) else ""
            acronym_hint = str(row["CDE Acronym Finder"]) if pd.notna(row["CDE Acronym Finder"]) else ""

            # ------------------------------------------------------------
            # Skip logic:
            # Do not call the LLM if the prestep response is empty or failed parsing.
            # This saves API calls and makes the audit trail cleaner.
            # ------------------------------------------------------------
            parse_status = str(row["Parse Status"]).strip() if "Parse Status" in working_df.columns and pd.notna(row["Parse Status"]) else ""

            if not full_response.strip():
                working_df.at[idx, "HEAL Core CRF Match"] = "No CRF match"
                working_df.at[idx, "Prestep CRF Confidence"] = ""
                working_df.at[idx, "Match Rationale"] = "Skipped because Full Response was empty."
                working_df.at[idx, "Raw HEAL Match Response"] = ""
                working_df.at[idx, "HEAL Match Status"] = "SKIPPED_EMPTY_PRESTEP"
                working_df.at[idx, "HEAL Match Attempts"] = 0
                working_df.at[idx, "HEAL Match Error"] = "Full Response was empty."
                continue

            if parse_status.upper() in ["PARSE_FAILED", "FAILED", "ERROR"]:
                working_df.at[idx, "HEAL Core CRF Match"] = "No CRF match"
                working_df.at[idx, "Prestep CRF Confidence"] = ""
                working_df.at[idx, "Match Rationale"] = "Skipped because prestep output could not be parsed."
                working_df.at[idx, "Raw HEAL Match Response"] = ""
                working_df.at[idx, "HEAL Match Status"] = "SKIPPED_PARSE_FAILED"
                working_df.at[idx, "HEAL Match Attempts"] = 0
                working_df.at[idx, "HEAL Match Error"] = f"Parse Status was {parse_status}."
                continue

            # ------------------------------------------------------------
            # Only rows that pass the skip checks are sent to the LLM.
            # ------------------------------------------------------------
            result = call_heal_match_with_retry(
                full_prestep_response=full_response,
                acronym_hint=acronym_hint,
                max_retries=3,
                base_sleep_seconds=2
            )

            working_df.at[idx, "HEAL Core CRF Match"] = result["HEAL Core CRF Match"]
            working_df.at[idx, "Prestep CRF Confidence"] = result["Prestep CRF Confidence"]
            working_df.at[idx, "Match Rationale"] = result["Match Rationale"]
            working_df.at[idx, "Raw HEAL Match Response"] = result["Raw HEAL Match Response"]
            working_df.at[idx, "HEAL Match Status"] = result["HEAL Match Status"]
            working_df.at[idx, "HEAL Match Attempts"] = result["HEAL Match Attempts"]
            working_df.at[idx, "HEAL Match Error"] = result["HEAL Match Error"]

    # Clean final output display:
    # If there is no HEAL Core CRF Match, blank out the prestep confidence field.
    mask_no_crf = working_df["HEAL Core CRF Match"].astype(str).str.strip().eq("No CRF match")
    working_df.loc[mask_no_crf, "Prestep CRF Confidence"] = ""

    return working_df


print("✅ Responses API HEAL Core CRF matcher loaded.")

✅ Responses API HEAL Core CRF matcher loaded.


In [ ]:
# ----------------------------
# Final orchestrator: acronym finder + prestep + harmonizer + HEAL match + save
# ----------------------------
full_input_df = pd.read_excel(input_file, sheet_name=input_worksheet).copy()

# Step 1: keep only the columns needed for the CRF workflow
data_dict_df = full_input_df[[crf_column, variable_column, description_column]].copy()

# Step 2: acronym finder
data_dict_df["CDE Acronym Finder"] = data_dict_df[variable_column].apply(detect_heal_cde_acronym)

print("\n[CDE Acronym Finder] Preview:")
print(data_dict_df[[variable_column, "CDE Acronym Finder"]].head(10).to_string(index=False))

# Step 3: prestep refinement via Responses API
prestep_checkpoint_file = Path(output_file).with_name(
    Path(output_file).stem + "_prestep_checkpoint.xlsx"
)

refined_df = run_prestep_responses(
    data_dict_df,
    chunk_size=20,
    checkpoint_every=20,
    checkpoint_path=prestep_checkpoint_file
)

print("\n[Prestep] Sample after refinement:")
print(
    refined_df[
        [crf_column, variable_column, "Refined CRF Name", "Rationale", "Prestep Run Status", "Parse Status"]
    ].head(10).to_string(index=False)
)

# Step 4: harmonize refined CRF names
harmonized_df = run_form_harmonizer(refined_df, batch_size=20)

print("\n[Harmonizer] Sample after canonical naming:")
print(
    harmonized_df[
        ["Refined CRF Name", "Canonical CRF Name"]
    ].drop_duplicates().head(20).to_string(index=False)
)

# Step 5: merge harmonized fields back into the full input
enhanced_df = full_input_df.join(
    harmonized_df[[
        "CDE Acronym Finder",
        "Full Response",
        "Refined CRF Name",
        "Rationale",
        "Parsed Full Response",
        "Parse Status",
        "Parse Error",
        "Prestep Run Status",
        "Prestep Attempts",
        "Prestep Error",
        "Canonical CRF Name"
    ]],
    how="left"
)

# Step 6: HEAL Core CRF match
final_df = run_heal_match(enhanced_df, chunk_size=20)

print("\n[HEAL Match] Sample after matching:")
print(
    final_df[
        [
            variable_column,
            "Refined CRF Name",
            "Canonical CRF Name",
            "HEAL Core CRF Match",
            "Prestep CRF Confidence",
            "Match Rationale",
            "HEAL Match Status"
        ]
    ].head(10).to_string(index=False)
)

# Step 7: metadata sheet
metadata_required_cols = [
    crf_column,
    "Canonical CRF Name",
    "HEAL Core CRF Match",
    "Match Rationale"
]

# Add blank columns if any expected metadata columns are missing.
# This prevents the save step from crashing and makes missing pieces visible.
for col in metadata_required_cols:
    if col not in final_df.columns:
        print(f"⚠️ Metadata column missing from final_df: {col}. Creating blank column.")
        final_df[col] = ""

metadata_df = (
    final_df[metadata_required_cols]
    .copy()
    .rename(columns={
        crf_column: "Original CRF Name",
        "Match Rationale": "Rationale"
    })
    .drop_duplicates(
        subset=["Original CRF Name", "Canonical CRF Name", "HEAL Core CRF Match"]
    )
    .reset_index(drop=True)
)

# Optional sorting for easier review
metadata_df = metadata_df.sort_values(
    by=["Original CRF Name", "Canonical CRF Name", "HEAL Core CRF Match"],
    na_position="last"
).reset_index(drop=True)

# Step 8: save final workbook
with pd.ExcelWriter(output_file, engine="xlsxwriter") as writer:
    metadata_df.to_excel(writer, sheet_name="Metadata", index=False)
    final_df.to_excel(writer, sheet_name="EnhancedDD", index=False)

print(f"\n✅ Results saved to {output_file}")
print("   Sheets written: 'Metadata' and 'EnhancedDD'")
print(f"   Checkpoint file: {prestep_checkpoint_file}")


[CDE Acronym Finder] Preview:
            name CDE Acronym Finder
       record_id                   
       subjectid                   
      oboe_group                   
study_identifier                   
     sc_birthdat                   
      esubjectid                   
      wsubjectid                   
        sccomnts                   
       languages                   
     opiodrx___1                   

Processing rows 0 to 19...
Checkpoint saved after 20 rows to: out\Testfile_2026-04-29_prestep_checkpoint.xlsx

Processing rows 20 to 39...
Checkpoint saved after 40 rows to: out\Testfile_2026-04-29_prestep_checkpoint.xlsx

Processing rows 40 to 59...
Checkpoint saved after 60 rows to: out\Testfile_2026-04-29_prestep_checkpoint.xlsx

Processing rows 60 to 79...
Checkpoint saved after 80 rows to: out\Testfile_2026-04-29_prestep_checkpoint.xlsx

Processing rows 80 to 99...
Checkpoint saved after 100 rows to: out\Testfile_2026-04-29_prestep_checkpoint.xlsx

Processing r